In [ ]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from ray import tune
from ray.tune.schedulers.pb2 import PB2, PopulationBasedTraining
from ray.tune import Checkpoint, run, sample_from 

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapperFocus import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = config.Config()

In [ ]:
class PGTrainer(object):
    def __init__(
        self, 
        env, 
        model, 
        optimizer, 
        scheduler, 
        gamma=0.99, 
        update_target_every=10
    ):
        self.env = env
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.gamma = gamma
        self.update_target_every = update_target_every
        self.n_step_buffer = []

In [ ]:
def rl(cfg):

    alg_name ="ATLA"
    env_name ="EVS"

    model = 

    trainer = PGTrainer(
        env=EnvWrapper(cfg),
        model=MultiHeadQNetwork(cfg.state_dim, cfg.action_dim_meta, cfg.action_dim_ctrl).to(device),
        optimizer=optim.Adam,
        scheduler=None,
        gamma=cfg.gamma,
        update_target_every=10
    )

    start_episode = 0
    checkpoint = tune.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            checkpoint_data = torch.load(
                os.path.join(checkpoint_dir, "checkpoint.pt"),
                map_location=device
            )

            start_episode = checkpoint_data["episode"] + 1
            trainer.behaviour_net.load_state_dict(checkpoint_data["model_state_dict"])
            print(f"== resume from checkpoint, continue with epoch {start_episode} \n")

        rewards = []

        for episode in range(cfg.n_episodes):
            reward = trainer.run(episode)
            rewards.append(reward)

            if episode % cfg.nb_interval == 0:
                checkpoint_data = {
                    "episode": episode,
                    "model_state_dict": trainer.behaviour_net.state_dict(),
                }
                with tune.checkpoint_dir(episode) as checkpoint_dir:
                    torch.save(checkpoint_data, os.path.join(checkpoint_dir, "checkpoint.pt"))

    for i in range(cfg.n_episodes):
        stat = {}
        hyperparams = {
            "n_step": cfg.n_step,
            "gamma": cfg.gamma,
            "lr": cfg.learning_rate,
            "batch_size": cfg.batch_size,
            "buffer_capacity": cfg.buffer_capacity,
        }
        trainer.run(stat, i, hyperparams=hyperparams)

In [ ]:
if __name__ == "__main__":
    pb2 = PB2(
        metric="objective",
        mode="max",
        quantile_fraction=0.25,
        perturbation_interval=160,
        hyperparam_bounds={
            "alpha": [0.05, 0.2],
            "beta": [0.3, 0.6]
        }
    )
    
    for seed in range(0, 4):
        analysis = tune.run(
            rl,
            scheduler=pb2,
            num_samples=4,
            reuse_actors=True,
            config={
                "alpha": sample_from(lambda spec: np.random.uniform(0.05, 0.2)),
                "beta": sample_from(lambda spec: np.random.uniform(0.4, 0.6)),
                "seed": seed
            }
        )
        
        all_dfs = analysis.trial_dataframes
        names = list(all_dfs.keys())
        
        results = pd.DataFrame()
        for i in range(4):
            df = all_dfs[names[i]].copy()
            df['sample_num'] = i 
            results = pd.concat([results, df]).reset_index(drop=True)

        dir = "{}_{}_{}_Size{}_{}_{}_{}_{}_{}".format(rl, "file", "method", str(4), "env", "default", "max", "160", "batch")
        exist_dir = os.path.expanduser('~/data/' + dir)
        if not(os.path.exists(exist_dir)):
            os.makedirs(exist_dir)

        result_dir1 = os.path.expanduser('~/data/')
        result_dir2 = f"{dir}/seed{seed}.csv"
        results.to_csv(result_dir1 + result_dir2)
        